In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import gc
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

In [2]:
#Import bitext comolaint dataset
bitext = pd.read_csv('/kaggle/input/datasets/bitext/bitext-gen-ai-chatbot-customer-support-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv', usecols=['instruction', 'intent']).rename(columns={'instruction':'text'})

#intent to expect category mapping 

BITEXT_TO_TARGET = {
    "cancel_order": "general_inquiry",
    "change_order": "general_inquiry",
    "change_shipping_address": "delivery_shipping",
    "check_cancellation_fee": "subscription_cancel",
    "check_invoice": "billing",
    "check_payment_methods": "billing",
    "check_refund_policy": "refund_return",
    "complaint": "customer_service",
    "contact_customer_service": "customer_service",
    "contact_human_agent": "customer_service",
    "create_account": "account_access",
    "delete_account": "account_access",
    "delivery_options": "delivery_shipping",
    "delivery_period": "delivery_shipping",
    "edit_account": "account_access",
    "get_invoice": "billing",
    "get_refund": "refund_return",
    "newsletter_subscription": "general_inquiry",
    "payment_issue": "billing",
    "place_order": "general_inquiry",
    "recover_password": "account_access",
    "registration_problems": "account_access",
    "review": "customer_service",
    "set_up_shipping_address": "delivery_shipping",
    "switch_account": "account_access",
    "track_order": "delivery_shipping",
    "track_refund": "refund_return"
}

bitext['label'] = bitext.intent.map(BITEXT_TO_TARGET)
display(bitext.head())
bitext.shape

,text,intent,label
0,question about cancelling order {{Order Number}},cancel_order,general_inquiry
1,i have a question about cancelling oorder {{Or...,cancel_order,general_inquiry
2,i need help cancelling puchase {{Order Number}},cancel_order,general_inquiry
3,I need to cancel purchase {{Order Number}},cancel_order,general_inquiry
4,"I cannot afford this order, cancel purchase {{...",cancel_order,general_inquiry


(26872, 3)

In [3]:
train = pd.read_csv('/kaggle/input/datasets/alexanderdape/prov-train/train_complaints.csv', usecols=['text', 'Category'])
train['label'] = train.Category
display(train.head())
train.shape

,text,Category,label
0,Flagging an issue: Username change broke acces...,account_access,account_access
1,Writing to complain: Blender motor smells like...,product_defect,product_defect
2,"Hello, Courier left perishable groceries in th...",delivery_shipping,delivery_shipping
3,Hi support — Tablet touchscreen registers ghos...,product_defect,product_defect
4,Hi support — Authorized service charged labor ...,warranty_repair,warranty_repair


(380, 3)

In [4]:
cfpb = pd.read_csv('/kaggle/input/datasets/alexanderdape/cfpb-fraud/cfpb_fraud_complaints-2026-08-24_05_35.csv', usecols=['Consumer complaint narrative', 'Issue']).rename(columns={'Consumer complaint narrative':'text'})
cfpb['label'] = 'fraud_unauthorized'
display(cfpb.head())
cfpb.shape

,Issue,text,label
0,Fraud or scam,I sent XXXX Ethereum from a crypto wallet and ...,fraud_unauthorized
1,Unauthorized transactions or other transaction...,Failure to Investigate Unauthorized Electronic...,fraud_unauthorized
2,Fraud or scam,"On XX/XX/year>, I deposited a money order into...",fraud_unauthorized
3,Unauthorized transactions or other transaction...,Through the evening of XX/XX/XXXX and into the...,fraud_unauthorized
4,Fraud or scam,"First, XX/XX/XXXX hard hit on credit by capita...",fraud_unauthorized


(1694, 3)

In [5]:
full_df = pd.concat([bitext[['text', 'label']],cfpb[['text', 'label']],train[['text', 'label']],], ignore_index=True).dropna(how='any')

display(full_df.label.value_counts())

label
account_access         6026
delivery_shipping      4999
billing                4047
general_inquiry        4027
customer_service       4026
refund_return          3032
subscription_cancel     985
fraud_unauthorized      623
product_defect           35
warranty_repair          35
Name: count, dtype: int64

In [6]:
# Split and Encode
train_df, val_df = train_test_split(full_df, test_size=0.15, stratify=full_df["label"], random_state=42)

le = LabelEncoder()
train_df["label_id"] = le.fit_transform(train_df["label"])
val_df["label_id"] = le.transform(val_df["label"])
joblib.dump(le, "label_encoder.pkl")
label_names = list(le.classes_)
num_labels = len(label_names)

# Class weights
class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_labels), y=train_df["label_id"].values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Move weights to the same device as logits dynamically
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    per_class = f1_score(labels, preds, average=None, labels=np.arange(num_labels))
    metrics = {"macro_f1": macro_f1}
    for name, score in zip(label_names, per_class):
        metrics[f"f1_{name}"] = score
    return metrics

In [7]:
MODEL_CHECKPOINTS = {
    "modernbert": "answerdotai/ModernBERT-base",
}

results = {}
SAVE_ROOT = "/kaggle/working/saved_models"
test_df = pd.read_csv("/kaggle/input/datasets/alexanderdape/prov-train/test_complaints.csv")

for model_key, MODEL_NAME in MODEL_CHECKPOINTS.items():
    print(f"\n{'='*50}\nTraining: {model_key} ({MODEL_NAME})\n{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Calculate 95th percentile dynamically; do NOT overwrite this with 64.
    lengths = train_df["text"].apply(lambda x: len(tokenizer.tokenize(x)))
    MAX_LEN = min(512, int(lengths.quantile(0.95)) + 10)
    MAX_LEN = 128
    print(f"Dynamic MAX_LEN for {model_key}: {MAX_LEN}")

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

    # Re-instantiate datasets inside the loop to avoid caching conflicts between different tokenizers
    train_ds_m = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
    val_ds_m = Dataset.from_pandas(val_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
    test_ds = Dataset.from_pandas(test_df[["text"]])

    train_ds_m = train_ds_m.map(tokenize_fn, batched=True)
    val_ds_m = val_ds_m.map(tokenize_fn, batched=True)
    test_ds = test_ds.map(tokenize_fn, batched=True)

    train_ds_m.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_ds_m.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    args = TrainingArguments(
        output_dir=f"./results_{model_key}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=5, 
        warmup_ratio=0.08,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        fp16=True # Enable mixed precision for memory efficiency
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds_m,
        eval_dataset=val_ds_m,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Validation Evaluation
    preds = trainer.predict(val_ds_m)
    y_pred = np.argmax(preds.predictions, axis=1)
    y_true = preds.label_ids
    report = classification_report(y_true, y_pred, target_names=label_names, digits=3, output_dict=True)
    results[model_key] = report
    print(classification_report(y_true, y_pred, target_names=label_names, digits=3))

    # Test Prediction and Submission
    test_preds = trainer.predict(test_ds)
    pred_ids = np.argmax(test_preds.predictions, axis=1)
    submission = pd.DataFrame({
        "ComplaintId": test_df["ComplaintId"],      
        "Category": le.inverse_transform(pred_ids)        
    })
    submission.to_csv(f"submission_{model_key}.csv", index=False)

    # ---- Save model + tokenizer + label encoder before cleanup ----
    save_path = os.path.join(SAVE_ROOT, model_key)
    os.makedirs(save_path, exist_ok=True)
    trainer.save_model(save_path)       # saves model weights + config
    tokenizer.save_pretrained(save_path) # save the matching tokenizer, not a shared one — MAX_LEN/vocab differs per model
    joblib.dump(le, os.path.join(save_path, "label_encoder.pkl"))

    print(f"Saved {model_key} to {save_path}")

    # Force garbage collection to prevent memory leaks across loops
    del model, trainer, train_ds_m, val_ds_m, test_ds
    torch.cuda.empty_cache()
    gc.collect()

print("\n\n=== Weak-class F1 comparison ===")
for model_key, report in results.items():
    pd_f1 = report.get("product_defect", {}).get("f1-score", "N/A")
    wr_f1 = report.get("warranty_repair", {}).get("f1-score", "N/A")
    macro = report["macro avg"]["f1-score"]
    print(f"{model_key}: product_defect={pd_f1:.3f}, warranty_repair={wr_f1:.3f}, macro_f1={macro:.3f}")


Training: modernbert (answerdotai/ModernBERT-base)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

                     precision    recall  f1-score   support

     account_access      0.999     1.000     0.999       904
            billing      1.000     1.000     1.000       607
   customer_service      1.000     1.000     1.000       604
  delivery_shipping      1.000     0.999     0.999       750
 fraud_unauthorized      1.000     1.000     1.000        94
    general_inquiry      1.000     1.000     1.000       604
     product_defect      1.000     1.000     1.000         5
      refund_return      1.000     1.000     1.000       455
subscription_cancel      1.000     1.000     1.000       148
    warranty_repair      1.000     1.000     1.000         5

           accuracy                          1.000      4176
          macro avg      1.000     1.000     1.000      4176
       weighted avg      1.000     1.000     1.000      4176



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved modernbert to /kaggle/working/saved_models/modernbert


=== Weak-class F1 comparison ===
modernbert: product_defect=1.000, warranty_repair=1.000, macro_f1=1.000
